# `financialtools.utils` — User Guide

I/O helpers, data-conversion utilities, and Yahoo Finance profile enrichment.

```
financialtools/utils.py
│
├── I/O Helpers
│   ├── export_to_csv(df, path)                     → None
│   └── export_to_xlsx(df, path, sheet_name)        → None
│
├── Weight Utilities
│   └── flatten_weights(weights)                     → dict
│
├── Data Conversion
│   └── dataframe_to_json(df)                        → str
│
└── Yahoo Finance Enrichment
    ├── get_ticker_profile(ticker)                   → pd.DataFrame (1 row)
    └── enrich_tickers(df, ticker_column)            → pd.DataFrame
```

| Function | Needs files on disk | Needs internet | Raises |
|---|---|---|---|
| `export_to_csv` | no | no | — (errors printed) |
| `export_to_xlsx` | no | no | — (errors printed) |
| `flatten_weights` | no | no | — (errors printed) |
| `dataframe_to_json` | no | no | `TypeError` |
| `get_ticker_profile` | no | **yes** (yfinance) | `ImportError` |
| `enrich_tickers` | no | **yes** (yfinance) | `ValueError` if all fail |

> **Removed functions** — `get_tickers`, `get_sector_for_ticker`, `get_market_metrics`,
> `get_fin_data`, `get_fin_data_year`, `list_evaluated_tickers`, `create_newbatch_folder`
> have been removed from the package. Sections 6–8 below are archived.

> **Sector weights** — use `financialtools.config.sector_metric_weights` directly:
> ```python
> from financialtools.config import sector_metric_weights
> import pandas as pd
> weights = (pd.DataFrame(list(sector_metric_weights["Technology"].items()),
>                         columns=["metrics", "weights"]).assign(sector="Technology"))
> ```

**Prerequisites:**
- Sections 1–5 run with **no API key and no network**.
- Section 6 (`get_ticker_profile`, `enrich_tickers`) requires internet access and `yfinance`.

## Sections
1. [Setup](#1--setup)
2. [Sample data](#2--sample-data)
3. [export_to_csv / export_to_xlsx](#3--export_to_csv--export_to_xlsx)
4. [flatten_weights](#4--flatten_weights)
5. [dataframe_to_json](#5--dataframe_to_json)
6. [Live: get_ticker_profile / enrich_tickers](#6--live-get_ticker_profile--enrich_tickers)
7. [Error handling reference](#7--error-handling-reference)
8. [End-to-end pattern](#8--end-to-end-pattern)

## 1 — Setup

In [ ]:
import json
import os
import tempfile

import pandas as pd

from financialtools.utils import (
    export_to_csv,
    export_to_xlsx,
    dataframe_to_json,
    flatten_weights,
    get_ticker_profile,
    enrich_tickers,
)

print("Imports OK")

## 2 — Sample data

A small in-memory DataFrame reused across Sections 3–5. No files on disk required.

In [ ]:
SAMPLE_DF = pd.DataFrame({
    "ticker":  ["AAPL", "MSFT", "GOOGL"],
    "sector":  ["Technology", "Technology", "Technology"],
    "score":   [4.2, 3.8, 4.0],
    "year":    [2023, 2023, 2023],
})
print(SAMPLE_DF)

## 3 — `export_to_csv` / `export_to_xlsx`

Both functions are thin pandas I/O wrappers.

**Signatures:**
```python
export_to_csv(df: pd.DataFrame, path: str) -> None
export_to_xlsx(df: pd.DataFrame, path: str, sheet_name: str) -> None
```

Key behaviours:
- `export_to_csv` writes without a row index (`index=False`).
- `export_to_xlsx` uses `openpyxl` as the engine; the sheet is named by `sheet_name`.
- Both functions **print** a confirmation on success. Exceptions are caught internally and printed rather than re-raised — check that the output file actually exists when correctness matters.

In [ ]:
with tempfile.TemporaryDirectory() as tmp:
    # --- CSV round-trip ---
    csv_path = os.path.join(tmp, "scores.csv")
    export_to_csv(SAMPLE_DF, csv_path)
    csv_back = pd.read_csv(csv_path)
    print(f"CSV round-trip OK : {csv_back.shape} rows×cols")
    print(csv_back.to_string(index=False))

    print()

    # --- Excel round-trip ---
    xlsx_path = os.path.join(tmp, "scores.xlsx")
    export_to_xlsx(SAMPLE_DF, xlsx_path, sheet_name="results")
    xlsx_back = pd.read_excel(xlsx_path, sheet_name="results")
    print(f"XLSX round-trip OK: {xlsx_back.shape} rows×cols")
    print(xlsx_back.to_string(index=False))

## 4 — `flatten_weights`

**Signature:**
```python
flatten_weights(weights: dict) -> dict
```

`config.py` stores scoring weights in a *grouped* format where top-level keys are category names and values are `{metric: weight}` dicts. `flatten_weights` collapses that one level of nesting into a single `{metric: weight}` dict that scoring functions expect.

Rules:
- If a value is a `dict` it is unpacked (grouped schema).
- If a value is a scalar it is passed through as-is (already-flat schema).
- Mixed inputs work: flat entries and grouped entries can coexist.
- On any exception the function prints the error and returns `{}` rather than raising.

In [ ]:
# --- Grouped input (the common case from config.py) ---
grouped = {
    "Profitability & Margins": {"GrossMargin": 10, "OperatingMargin": 12},
    "Return":                  {"ROA": 10, "ROE": 12},
    "Leverage":                {"DebtToEquity": 8},
}
flat = flatten_weights(grouped)
print("Grouped -> flat:")
for k, v in flat.items():
    print(f"  {k:<20} : {v}")

print()

# --- Already-flat input passes through unchanged ---
already_flat = {"GrossMargin": 10, "ROA": 6, "DebtToEquity": 8}
print("Already flat -> flat:", flatten_weights(already_flat))

print()

# --- Mixed input ---
mixed = {
    "Profitability": {"GrossMargin": 10},   # grouped
    "FCFYield": 8,                           # flat
}
print("Mixed -> flat:", flatten_weights(mixed))

## 5 — `dataframe_to_json`

**Signature:**
```python
dataframe_to_json(df: pd.DataFrame) -> str
```

Serialises a pandas DataFrame to a JSON string in `records` orientation (list of row dicts). The result can be embedded directly in an LLM prompt.

Raises `TypeError` if the input is not a `pd.DataFrame`.

In [ ]:
# --- Happy path ---
json_str = dataframe_to_json(SAMPLE_DF)
print("Type  :", type(json_str))
print("Output:", json_str)

print()

# --- Verify it is valid JSON ---
parsed = json.loads(json_str)
print(f"Parsed : {len(parsed)} record(s), first = {parsed[0]}")

## ~~6 — `get_tickers`~~ (removed)

> `get_tickers` and `get_sector_for_ticker` have been removed from the package.
> They depended on a bundled `sector_ticker.txt` data file that is no longer
> distributed with the package.
>
> If you need to map tickers to sectors, maintain your own lookup and pass
> `sector` explicitly to `run_topic_analysis()` or `get_stock_evaluation_report()`.

## 6 — Live: `get_ticker_profile` / `enrich_tickers`

> **LIVE cells** — require internet access and `yfinance`.

**Signatures:**
```python
get_ticker_profile(ticker: str) -> pd.DataFrame   # 1-row DataFrame
enrich_tickers(df: pd.DataFrame, ticker_column: str = "ticker") -> pd.DataFrame
```

`get_ticker_profile` fetches key profile fields from yfinance: `shortName`, `industry`,
`sectorKey`, `beta`, `longBusinessSummary`. One row per ticker.

`enrich_tickers` applies `get_ticker_profile` to every ticker in a DataFrame column,
sleeps 0.5 s between calls to avoid rate limits, and combines results. Failed tickers
are printed and skipped — the rest still return.

In [ ]:
# LIVE — requires internet and yfinance

profile = get_ticker_profile("AAPL")
print(profile[["shortName", "industry", "sectorKey", "beta"]].to_string(index=False))

print()

# Enrich a small ticker list
tickers_df = pd.DataFrame({"ticker": ["AAPL", "MSFT"]})
enriched = enrich_tickers(tickers_df, ticker_column="ticker")
print(enriched[["ticker", "shortName", "sectorKey"]].to_string(index=False))

## ~~7 — Live: `list_evaluated_tickers`~~ (removed)

> `list_evaluated_tickers`, `get_market_metrics`, `get_fin_data`, and `get_fin_data_year`
> have been removed from the package. They read from pre-computed Excel files in
> `financial_data/` which are no longer part of the package API.
>
> Use `run_topic_analysis(ticker, sector)` from `financialtools.analysis` instead —
> it is fully self-contained and does not require any pre-computed files.

In [ ]:
# These functions have been removed. See note above.
# Use run_topic_analysis() for self-contained analysis:
#
# from financialtools.analysis import run_topic_analysis
# result = run_topic_analysis("AAPL", sector="Technology", year=2023)
print("Section archived — functions removed from package.")